# Rebuild_taxonomy_master_final

In [ ]:
# build_taxonomy_master_final.py
#
# Builds a clean, complete taxonomy master from scratch:
#   - cat_keys.csv         authoritative DB key list (109 keys) - source of truth
#   - taxonomy_master_updated_old.csv  manually curated enrichment (spend_type,
#                          category_group, category_type, leans, manual_spend_override)
#
# Output: taxonomy_master_final.csv  (109 rows, no invalid keys, no gaps)

import pandas as pd
from pathlib import Path

CAT_KEYS_PATH = Path("assets/cat_keys.csv")
OLD_TAXONOMY  = Path("assets/taxonomy_master_updated_old.csv")
OUTPUT_PATH   = Path("assets/taxonomy_master_final.csv")









In [ ]:
# ============================================================
# 1. Load files
# ============================================================
cat_keys = pd.read_csv(CAT_KEYS_PATH)
old_tax  = pd.read_csv(OLD_TAXONOMY)

# Normalise key column - cat_keys uses base_category_key after 09 Mar rename
key_col = "base_category_key" if "base_category_key" in cat_keys.columns else "key"
cat_keys = cat_keys[[key_col]].rename(columns={key_col: "base_category_key"})
cat_keys["base_category_key"] = cat_keys["base_category_key"].str.strip().str.upper()

# Normalise old taxonomy key column
if "category_key" in old_tax.columns:
    old_tax = old_tax.rename(columns={"category_key": "base_category_key"})
old_tax["base_category_key"] = old_tax["base_category_key"].str.strip().str.upper()

print(f"cat_keys rows     : {len(cat_keys)}")
print(f"old taxonomy rows : {len(old_tax)}")

In [ ]:
# ============================================================
# 2. Left join - cat_keys drives, old taxonomy enriches
# ============================================================
merged = cat_keys.merge(
    old_tax[["base_category_key", "spend_type", "leans",
             "manual_spend_override", "category_type", "category_group", "notes"]],
    on="base_category_key",
    how="left"
)

missing = merged[merged["spend_type"].isna()]["base_category_key"].tolist()
print(f"\nKeys needing manual enrichment ({len(missing)}):")
for k in sorted(missing):
    print(f"  {k}")


In [ ]:
# ============================================================
# 3. Fill in the 17 missing keys
#    Basis: production taxonomy knowledge + PO decisions logged in discussion docs
#    Keys marked PO_REVIEW need confirmation before rerun
# ============================================================
FILL = {
    # spend keys - straightforward
    "AUTOMOTIVE":               ("spend",     None, None, None, "Transport",        None),
    "GIFTS":                    ("spend",     None, None, None, "Other",            None),
    "GYMS___FITNESS":           ("spend",     None, None, None, "Health",           None),
    "OFFICE_MAINTENANCE":       ("spend",     None, None, None, "Business",         None),
    "PERSONAL":                 ("spend",     None, None, None, "Other",            None),
    "PETS_PET_CARE":            ("spend",     None, None, None, "Other",            None),
    "RECREATION":               ("spend",     None, None, None, "Entertainment",    None),
    "RENT_PAID":                ("spend",     None, None, None, "Home",             None),
    "RESTAURANT":               ("spend",     None, None, None, "Food & Drink",     None),
    "TRAVEL_HOLIDAYS":          ("spend",     None, None, None, "Entertainment",    None),
    # non-spend fee keys - consistent with other fee keys in taxonomy
    "FOREIGN_FEES":             ("non_spend", None, None, "FEES", None,            None),
    "LATE_FEES":                ("non_spend", None, None, "FEES", None,            None),
    "OTHER_BILLS":              ("non_spend", None, None, "FEES", None,            None),
    "OVERDRAWN_FEES":           ("non_spend", None, None, "FEES", None,            None),
    "OVERLIMIT_FEES":           ("non_spend", None, None, "FEES", None,            None),
    "REBATE_FEES":              ("non_spend", None, None, "FEES", None,            None),
    # ambiguous - needs PO confirmation
    "TRANSFER_BETWEEN_ACCOUNTS":("non_spend", None, None, "TRANSFERS", None,       "PO_REVIEW - confirm spend_type"),
}

cols = ["spend_type", "leans", "manual_spend_override", "category_type", "category_group", "notes"]

for key, vals in FILL.items():
    mask = merged["base_category_key"] == key
    if mask.sum() == 0:
        print(f"WARNING: {key} not found in merged df - check spelling")
        continue
    for col, val in zip(cols, vals):
        if val is not None:
            merged.loc[mask, col] = val


In [ ]:
# ============================================================
# 4. Final validation
# ============================================================
print("\n" + "=" * 55)
print("VALIDATION")
print("=" * 55)

still_missing = merged[merged["spend_type"].isna()]["base_category_key"].tolist()
if still_missing:
    print(f"WARNING - spend_type still missing for: {still_missing}")
else:
    print("All 109 keys have spend_type - PASSED")

po_review = merged[merged["notes"].str.contains("PO_REVIEW", na=False)]
if not po_review.empty:
    print(f"\nKeys flagged for PO review before rerun:")
    print(po_review[["base_category_key", "notes"]].to_string(index=False))

print(f"\nFinal row count  : {len(merged)}")
print(f"spend_type breakdown:")
print(merged["spend_type"].value_counts().to_string())

In [ ]:
# ============================================================
# 5. Save
# ============================================================
merged.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")
print("\nDone - update Notebook 5 to load taxonomy_master_final.csv")